# RS-003 임베딩 모델 벤치마킹

REQ-002 SRS의 RS-003(임베딩 및 Cloud SQL 적재)은 `bge-m3 / ko-sroberta / OpenAI text-embedding-3` 세 후보를 비교 후 확정하도록 요구한다.

**이 노트북에서 실제로 비교한 것**
- `jhgan/ko-sroberta-multitask` (sentence-transformers, 로컬)
- `BAAI/bge-m3` (sentence-transformers, 로컬)

**OpenAI text-embedding-3는 제외**했다 — 팀 `.env`에 OpenAI API 키가 없고, 이미 Gemini 키만 발급된 상태라 새 유료 API 키 발급 없이는 비교 자체가 불가능하다. 필요하면 추후 키 발급 후 별도로 비교.

**주의**: 아직 RS-005의 정식 골든셋(20~30개, 팀 공동 라벨링)이 없다. 여기서는 임시로 만든 AI 개념 질의 4개로 정성적 sanity check만 수행했다. 정식 Recall@k 평가는 골든셋이 만들어진 뒤 별도로 진행해야 한다.

In [ ]:
import json
import time
from pathlib import Path

from sentence_transformers import SentenceTransformer, util

DATA_PATH = Path("../app/data/curriculum_units.json")

CANDIDATES = [
    "jhgan/ko-sroberta-multitask",
    "BAAI/bge-m3",
]

# 정식 골든셋(RS-005) 이전 임시 sanity-check 질의. AI 개념 + A1이 만들 법한 재작성 정의를 흉내냈다.
QUERIES = [
    ("분류", "주어진 대상을 공통된 특징이나 기준에 따라 몇 가지 범주로 나누는 것"),
    ("패턴 인식", "자료나 사물의 배열에서 일정하게 반복되거나 변화하는 규칙을 찾아내는 것"),
    ("군집화", "정답 범주가 미리 주어지지 않은 상태에서 비슷한 대상끼리 스스로 묶는 것"),
    ("측정과 비교", "여러 대상의 속성을 수치화된 단위로 나타내어 서로 비교하는 것"),
]


def embedding_source_text(chunk: dict) -> str:
    return f"{chunk['unit_name']} {chunk['core_idea']} {chunk['achievement_text']} {chunk['explanation']}".strip()

In [ ]:
chunks = json.loads(DATA_PATH.read_text(encoding="utf-8"))
texts = [embedding_source_text(c) for c in chunks]
print(f"청크 수: {len(chunks)}")

청크 수: 223


## 모델별 로드/임베딩 속도 + 질의 5개 top-5 결과

223개 청크(수학 121 + 과학 102, `app/agents/curriculum_search`에서 파싱한 실제 데이터) 전체를 임베딩하고, 위 4개 질의로 코사인 유사도 top-5를 뽑는다.

In [ ]:
def benchmark(model_name: str) -> dict:
    print(f"\n{'=' * 60}\n모델: {model_name}\n{'=' * 60}")

    t0 = time.time()
    model = SentenceTransformer(model_name)
    load_time = time.time() - t0

    t0 = time.time()
    chunk_embeddings = model.encode(texts, show_progress_bar=False, convert_to_tensor=True)
    embed_time = time.time() - t0
    dim = chunk_embeddings.shape[1]

    print(f"로드 시간: {load_time:.2f}s")
    print(f"전체 {len(texts)}개 청크 임베딩 시간: {embed_time:.2f}s ({embed_time / len(texts) * 1000:.1f}ms/청크)")
    print(f"임베딩 차원: {dim}")

    query_results = {}
    for concept, definition in QUERIES:
        query_embedding = model.encode(definition, convert_to_tensor=True)
        scores = util.cos_sim(query_embedding, chunk_embeddings)[0]
        top5 = scores.topk(5)
        hits = []
        for score, idx in zip(top5.values.tolist(), top5.indices.tolist()):
            c = chunks[idx]
            hits.append((round(score, 4), c["subject"], c["unit_name"], c["achievement_code"], c["achievement_text"][:40]))
        query_results[concept] = hits
        print(f"\n  질의: {concept} ({definition})")
        for score, subject, unit, code, text in hits:
            print(f"    [{score}] {subject} {unit} {code} {text}...")

    return {
        "load_time": load_time,
        "embed_time": embed_time,
        "per_chunk_ms": embed_time / len(texts) * 1000,
        "dim": dim,
        "queries": query_results,
    }

In [ ]:
results = {}
results["jhgan/ko-sroberta-multitask"] = benchmark("jhgan/ko-sroberta-multitask")


모델: jhgan/ko-sroberta-multitask
로드 시간: 4.98s
전체 223개 청크 임베딩 시간: 1.99s (8.9ms/청크)
임베딩 차원: 768

  질의: 분류 (주어진 대상을 공통된 특징이나 기준에 따라 몇 가지 범주로 나누는 것)
    [0.6637] MATH 도형과 측정 [4수03-10] 여러 가지 모양의 사각형에 대한 분류 활동을 통하여 직사각형, 정사각형,...
    [0.6576] MATH 도형과 측정 [2수03-05] 삼각형, 사각형에서 각각의 공통점을 찾아 말할 수 있다....
    [0.6525] MATH 도형과 측정 [2수03-10] 길이 단위 1cm와 1m를 알고, 이를 이용하여 주변 사물의 길이를 측정...
    [0.6525] MATH 도형과 측정 [4수03-15] 길이 단위 1mm와 1km를 알고, 이를 이용하여 길이를 측정하고 어림하...
    [0.6496] MATH 도형과 측정 [4수03-20] 실생활에서 무게를 나타낼 때 사용하는 단위 1g과 1kg을 알고, 이를 ...

  질의: 패턴 인식 (자료나 사물의 배열에서 일정하게 반복되거나 변화하는 규칙을 찾아내는 것)
    [0.6891] MATH 변화와 관계 [2수02-01] 물체, 무늬, 수 등의 배열에서 규칙을 찾아 여러 가지 방법으로 표현할 ...
    [0.6891] MATH 변화와 관계 [2수02-02] 자신이 정한 규칙에 따라 물체, 무늬, 수 등을 배열할 수 있다....
    [0.6891] MATH 변화와 관계 [4수02-01] 다양한 변화 규칙을 찾아 설명하고, 그 규칙을 수나 식으로 나타낼 수 있...
    [0.6891] MATH 변화와 관계 [4수02-02] 계산식의 배열에서 규칙을 찾고, 계산 결과를 추측할 수 있다....
    [0.6891] MATH 변화와 관계 [4수02-03] 등호를 사용하여 크기가 같은 두 양의 관계를 식으로 나타낼 수 있다....

  질의: 군집화 (정답 범주가 미리 주어지지 않은 상

In [ ]:
results["BAAI/bge-m3"] = benchmark("BAAI/bge-m3")


모델: BAAI/bge-m3
로드 시간: 7.03s
전체 223개 청크 임베딩 시간: 26.00s (116.6ms/청크)
임베딩 차원: 1024

  질의: 분류 (주어진 대상을 공통된 특징이나 기준에 따라 몇 가지 범주로 나누는 것)
    [0.5371] MATH 변화와 관계 [2수02-01] 물체, 무늬, 수 등의 배열에서 규칙을 찾아 여러 가지 방법으로 표현할 ...
    [0.5365] MATH 변화와 관계 [2수02-02] 자신이 정한 규칙에 따라 물체, 무늬, 수 등을 배열할 수 있다....
    [0.5312] MATH 변화와 관계 [4수02-02] 계산식의 배열에서 규칙을 찾고, 계산 결과를 추측할 수 있다....
    [0.53] SCIENCE 물체와 물질 [4과05-01] 물체를 이루는 여러 가지 물질의 성질을 비교하고, 물질의 종류에 따라 물...
    [0.5288] MATH 변화와 관계 [4수02-01] 다양한 변화 규칙을 찾아 설명하고, 그 규칙을 수나 식으로 나타낼 수 있...

  질의: 패턴 인식 (자료나 사물의 배열에서 일정하게 반복되거나 변화하는 규칙을 찾아내는 것)
    [0.6645] MATH 변화와 관계 [2수02-01] 물체, 무늬, 수 등의 배열에서 규칙을 찾아 여러 가지 방법으로 표현할 ...
    [0.6401] MATH 변화와 관계 [2수02-02] 자신이 정한 규칙에 따라 물체, 무늬, 수 등을 배열할 수 있다....
    [0.6098] MATH 변화와 관계 [4수02-02] 계산식의 배열에서 규칙을 찾고, 계산 결과를 추측할 수 있다....
    [0.5901] MATH 변화와 관계 [4수02-01] 다양한 변화 규칙을 찾아 설명하고, 그 규칙을 수나 식으로 나타낼 수 있...
    [0.5873] MATH 변화와 관계 [4수02-03] 등호를 사용하여 크기가 같은 두 양의 관계를 식으로 나타낼 수 있다....

  질의: 군집화 (정답 범주가 미리 주어지지 않은 상태에서 비슷한 대상끼리

## 결과 요약

| 모델 | 로드 시간 | 임베딩 속도 (223개 청크) | 청크당 평균 | 차원 |
| --- | --- | --- | --- | --- |
| ko-sroberta-multitask | 4.98s | 1.99s | 8.9ms | 768 |
| bge-m3 | 7.03s | 26.00s | 116.6ms | 1024 |

**질의별 정성 비교 (상위 1개 유사도 점수, sanity check 4개)**

| 질의 | ko-sroberta top1 | bge-m3 top1 |
| --- | --- | --- |
| 분류 | 0.664 (사각형 분류, 도형과 측정) | 0.537 (규칙 배열, 변화와 관계) |
| 패턴 인식 | 0.689 (규칙 찾기, 변화와 관계) | 0.665 (규칙 찾기, 변화와 관계) |
| 군집화 | 0.528 (규칙 찾기로 대체) | 0.492 (규칙 찾기로 대체) |
| 측정과 비교 | 0.770 (길이·무게 비교, 도형과 측정) | 0.582 (규칙 배열, 변화와 관계 — 오답) |

### 결론

**ko-sroberta-multitask를 1단계 확정 모델로 채택**한다 (`app/agents/curriculum_search/logic.py`에 이미 반영).

- **속도**: ko-sroberta가 bge-m3보다 약 13배 빠르다 (8.9ms vs 116.6ms/청크). 데이터가 커지거나 재적재가 잦아지면 이 차이가 누적된다.
- **품질**: 4개 질의 중 3개(분류/패턴 인식/측정과 비교)에서 ko-sroberta가 더 높은 유사도 점수와 더 적절한 상위 결과를 냈다. 특히 "측정과 비교" 질의에서 bge-m3는 엉뚱하게 "변화와 관계" 영역을 1위로 꼽아 명백히 더 나쁜 결과를 냈다.
- **비용/차원**: bge-m3는 1024차원으로 ko-sroberta(768차원)보다 저장 공간을 33% 더 쓴다.
- **"군집화"** 질의는 두 모델 다 낮은 점수(0.49~0.53)로 "규칙 찾기"를 억지로 1위에 올렸다 — 실제로 초등 교육과정에 군집화에 대응하는 성취기준이 없다는 뜻일 가능성이 높다(도메인 갭, SRS 리스크 항목과 일치). RS-004 설계상 "관련 단원 없음"으로 명시 반환하는 임계치(threshold) 설정이 필요해 보인다 — 현재 `hybrid_search`는 top_k만큼 무조건 반환하므로 후속 논의 필요.

### 한계 (다음 단계에서 보완 필요)

1. **정식 골든셋 없음**: 이 비교는 RS-005가 요구하는 20~30개 팀 공동 라벨링 골든셋이 아니라, 4개 즉석 질의로 만든 sanity check다. Recall@15 같은 정량 지표는 골든셋이 만들어진 뒤 다시 측정해야 한다.
2. **OpenAI text-embedding-3 미비교**: API 키가 없어 비교 자체를 못했다. 필요하면 키 발급 후 이 노트북에 세 번째 후보로 추가.
3. **`inquiry_activities` 필드가 임베딩에 미포함**: 현재 `embedding_source_text`는 `unit_name + core_idea + achievement_text + explanation`만 사용하고 탐구 활동 텍스트는 빼고 있다. 실제로 도움이 될 수 있는 정보라 포함 여부는 팀 논의 필요.